# Laboratorio #4 — Fundamentos de Aprendizaje por Refuerzo y Gymnasium

**CC3092 Deep Learning y Sistemas Inteligentes**

Este notebook contiene (1) la investigación de los fundamentos de RL, (2) la
investigación de la librería Gymnasium y (3) un módulo exploratorio con la API
de Gymnasium: agente aleatorio en dos entornos y una política simple no
aprendida para CartPole-v1.

## 1. Fundamentos del aprendizaje por refuerzo

Investigación de los conceptos fundamentales. Fuentes citadas por tema.

**Referencias usadas en esta sección**

- [SB] R. Sutton y A. Barto, *Reinforcement Learning: An Introduction*, 2.ª ed.,
  MIT Press, 2018. <http://incompleteideas.net/book/the-book-2nd.html>
- [GYM] Gymnasium documentation — *Introduction / Basic Usage*.
  <https://gymnasium.farama.org/introduction/basic_usage/>
- [WD] C. Watkins y P. Dayan, "Q-learning", *Machine Learning* 8, 279–292, 1992.
  <https://doi.org/10.1007/BF00992698>

### ¿Qué es el aprendizaje por refuerzo y en qué se diferencia del supervisado y del no supervisado?

El aprendizaje por refuerzo (RL) es el problema de aprender **qué hacer** —cómo
mapear situaciones a acciones— para maximizar una señal escalar de recompensa
acumulada a lo largo del tiempo. El aprendiz no recibe ejemplos de la acción
correcta: debe descubrirla probando y observando la recompensa que produce
[SB, cap. 1].

- **vs. supervisado:** el supervisado aprende de un conjunto de ejemplos
  etiquetados `(entrada, salida correcta)` provistos por un supervisor externo;
  su objetivo es generalizar a entradas no vistas. En RL no hay etiqueta de la
  acción óptima, solo una recompensa que evalúa la acción tomada (feedback
  *evaluativo*, no *instructivo*), y las decisiones afectan qué datos se
  observarán después [SB, §1.1].
- **vs. no supervisado:** el no supervisado busca estructura oculta en datos sin
  etiqueta (clustering, densidad). RL tampoco usa etiquetas, pero su meta no es
  encontrar estructura sino **maximizar recompensa**. Es un tercer paradigma
  [SB, §1.1].
- Rasgos propios de RL: compromiso **exploración vs. explotación**, feedback
  **retardado** (una acción puede afectar recompensas muchos pasos después) y
  datos **no i.i.d.** que dependen de la política del agente.

### Componentes principales de un problema de RL

- **Agente:** el que aprende y decide. Observa el estado, elige acciones y
  recibe recompensas [GYM].
- **Entorno (environment):** todo lo externo al agente; recibe la acción y
  devuelve el siguiente estado y la recompensa. En Gymnasium es el objeto `Env`
  con `reset()` y `step()` [GYM].
- **Estado (state) / observación:** descripción de la situación actual usada por
  el agente para decidir. La *observación* puede ser parcial respecto al estado
  interno real del entorno.
- **Acción (action):** decisión del agente en cada paso; puede ser discreta
  (empujar izquierda/derecha) o continua (torque).
- **Recompensa (reward):** escalar `R_t` que emite el entorno tras cada acción;
  define **qué** se quiere lograr, no **cómo**. El objetivo del agente es
  maximizar el retorno `G_t = R_{t+1} + γR_{t+2} + γ²R_{t+3} + …` [SB, §3.1–3.3].
- **Política (policy) `π`:** la estrategia del agente, un mapeo de estados a
  acciones (`π(s)`) o a una distribución sobre acciones (`π(a|s)`). Es lo que se
  aprende [SB, §3.1].

### ¿Qué es un Proceso de Decisión de Markov (MDP)?

Un MDP es la formalización matemática de la toma de decisiones secuencial donde
las acciones influyen en recompensas inmediatas y en estados futuros. Cumple la
**propiedad de Markov**: el futuro depende solo del estado y acción actuales, no
de la historia completa [SB, cap. 3]. Un MDP finito es la tupla
`(S, A, P, R, γ)`:

- **`S` — estados:** conjunto de situaciones posibles del entorno.
- **`A` — acciones:** conjunto de decisiones disponibles (`A(s)` si depende del
  estado).
- **`P` — función de transición:** `P(s' | s, a)` = probabilidad de pasar a `s'`
  dado que en `s` se tomó `a`. Codifica la dinámica del entorno.
- **`R` — función de recompensa:** recompensa esperada `R(s, a, s')` (o
  `R(s, a)`) por esa transición.
- **`γ` — factor de descuento** (`0 ≤ γ ≤ 1`): pondera cuánto valen las
  recompensas futuras frente a las inmediatas. `γ→0` = agente miope; `γ→1` =
  agente previsor. Garantiza que el retorno sea finito en tareas continuas
  [SB, §3.3–3.4].

### Función de valor de estado `V(s)` y función de valor acción-estado `Q(s,a)`

- **`V^π(s)`** = retorno esperado si el agente **empieza en `s` y luego sigue la
  política `π`**: `V^π(s) = E_π[G_t | S_t = s]`. Mide "qué tan bueno" es un
  estado bajo `π` [SB, §3.5].
- **`Q^π(s,a)`** = retorno esperado si en `s` se toma la acción `a` (posiblemente
  distinta de `π(s)`) y **después** se sigue `π`:
  `Q^π(s,a) = E_π[G_t | S_t = s, A_t = a]`. Mide "qué tan buena" es una acción en
  un estado [SB, §3.5].
- **Relación entre ambas:** `V^π(s) = Σ_a π(a|s) Q^π(s,a)`.
- **Relación con la política óptima:** la política óptima `π*` maximiza el valor
  en todo estado. Sus funciones de valor `V*` y `Q*` satisfacen
  `V*(s) = max_a Q*(s,a)`, y una política óptima se obtiene actuando **greedy**
  sobre `Q*`: `π*(s) = argmax_a Q*(s,a)`. Conocer `Q*` basta para actuar
  óptimamente **sin modelo** del entorno; con solo `V*` se necesita `P` para
  mirar un paso adelante [SB, §3.6].

### Ecuación de Bellman

**Intuición:** el valor de un estado se puede escribir en términos del valor de
sus estados sucesores. "El valor de donde estoy = recompensa inmediata esperada
+ valor descontado de a dónde llego". Es una condición de **consistencia**
recursiva entre valores vecinos [SB, §3.5].

Ecuación de Bellman para `V^π`:

`V^π(s) = Σ_a π(a|s) Σ_{s'} P(s'|s,a) [ R(s,a,s') + γ V^π(s') ]`

Ecuación de **optimalidad** de Bellman (para `π*`):

`V*(s) = max_a Σ_{s'} P(s'|s,a) [ R(s,a,s') + γ V*(s') ]`

**Rol en el cálculo de las funciones de valor:** convierte el cálculo del valor
—que en principio es una suma infinita sobre trayectorias— en un sistema de
ecuaciones acopladas, una por estado. Los métodos de RL son esencialmente
formas de **resolver o aproximar** estas ecuaciones: programación dinámica
(barridos completos con el modelo), Monte Carlo (promedios de retornos
muestreados) y diferencia temporal / Q-Learning (actualizaciones incrementales
que empujan la estimación hacia el lado derecho de Bellman) [SB, cap. 4 y 6].

### Dilema exploración vs. explotación

Para maximizar recompensa el agente debe **explotar** lo que ya sabe que
funciona; pero para descubrir mejores acciones debe **explorar** alternativas
que parecen peores. Ninguna de las dos por sí sola tiene éxito, y explorar
cuesta recompensa a corto plazo [SB, §1.1, cap. 2]. Dos estrategias:

1. **ε-greedy:** con probabilidad `1−ε` se toma la acción greedy
   (`argmax_a Q(s,a)`), y con probabilidad `ε` una acción **uniforme al azar**.
   `ε` suele decaer con el tiempo (mucho explorar al inicio, casi puro explotar
   al final). Simple y muy usada; explora "a ciegas" sin distinguir entre la
   segunda mejor acción y la peor [SB, §2.2].
2. **Softmax / Boltzmann:** se muestrea la acción de una distribución
   `π(a|s) = exp(Q(s,a)/τ) / Σ_b exp(Q(s,b)/τ)`. La **temperatura `τ`** controla
   la aleatoriedad: `τ` alta → casi uniforme (mucha exploración); `τ` baja →
   casi greedy. A diferencia de ε-greedy, explora **proporcionalmente al valor
   estimado** (prueba más las acciones prometedoras) [SB, §2.3].

### Diferencias entre familias de métodos

- **Tareas episódicas vs. continuas:** las **episódicas** tienen un estado
  terminal que reinicia el problema (una partida, un intento de CartPole hasta
  caer); el retorno es una suma finita. Las **continuas** no terminan
  (control de un proceso 24/7); requieren `γ<1` (o recompensa media) para que el
  retorno sea finito [SB, §3.3–3.4]. En Gymnasium esto aparece como
  `terminated` (fin natural del episodio) vs. `truncated` (corte artificial por
  límite de tiempo).
- **On-policy vs. off-policy:** los métodos **on-policy** evalúan y mejoran la
  **misma** política que usan para generar los datos (p. ej. SARSA). Los
  **off-policy** aprenden sobre una política *objetivo* (normalmente la greedy /
  óptima) usando datos generados por otra política *de comportamiento* más
  exploratoria (p. ej. Q-Learning). Off-policy es más flexible (permite reusar
  datos, aprender de demostraciones) pero suele tener más varianza [SB, §5.5, 6.4–6.5].
- **Model-based vs. model-free:** **model-based** aprende o dispone de un modelo
  del entorno (`P` y `R`) y **planifica** con él (value/policy iteration, Dyna,
  MCTS); es más eficiente en muestras pero sensible a errores del modelo.
  **Model-free** (Q-Learning, SARSA, policy gradients) aprende valores o política
  **directamente de la experiencia**, sin modelo; más simple y robusto, pero
  necesita más interacción [SB, cap. 8].

### Q-Learning: actualización de valores y papel de `α` y `γ`

Q-Learning [WD] es un método **model-free, off-policy** de diferencia temporal
que aprende `Q*` directamente. Tras observar `(s, a, r, s')` actualiza:

`Q(s,a) ← Q(s,a) + α [ r + γ · max_{a'} Q(s',a') − Q(s,a) ]`

El término entre corchetes es el **error de diferencia temporal (TD error)**: la
diferencia entre la estimación actual `Q(s,a)` y un objetivo mejor informado
`r + γ·max_{a'} Q(s',a')` (recompensa real observada + mejor valor futuro
estimado). La actualización mueve `Q(s,a)` una fracción `α` hacia ese objetivo.
Es **off-policy** porque el objetivo usa `max_{a'}` (política greedy) aunque la
acción `a` se haya elegido explorando (p. ej. ε-greedy).

- **Tasa de aprendizaje `α` (`0<α≤1`):** tamaño del paso. `α` alta → aprende
  rápido pero las estimaciones oscilan con el ruido de las transiciones; `α`
  baja → convergencia lenta pero estable. La convergencia teórica a `Q*` exige
  que `α` decaiga adecuadamente y que todo par `(s,a)` se visite infinitamente
  [WD; SB, §6.5].
- **Factor de descuento `γ` (`0≤γ<1`):** cuánto pesa `max_{a'} Q(s',a')`, es
  decir el futuro. `γ` cerca de 0 hace al agente miope (solo busca `r`
  inmediata); `γ` cerca de 1 lo hace planificar a largo plazo, pero ralentiza la
  propagación de la señal y puede inestabilizar el aprendizaje con aproximación
  de funciones [SB, §3.4, 6.5].